# 🐦 Scraping Twitter/X — **Tweet Harvest Engine**
> **Portfolio Project:** High-Speed Social Media Ingestion via Token-Based CLI Engine  
> **Topic:** Geopolitical Intelligence — US–Iran WW3 Escalation Discourse  
> **Target Output:** `data/raw/`  
> **Keyword:** `("world war" OR ww3 OR wwiii) (iran OR tehran) (america OR us OR usa OR washington) lang:en -filter:retweets`

---
## 📋 Pipeline Workflow
1. [System & CLI Environment Check](#1-persiapan--instalasi)
2. [Interactive Authentication Setup](#2-login--cookie-setup)
3. [Batch Scraping Execution](#3-scraping-tweet)
4. [Data Validation & Ingestion](#4-load--validasi-data)
5. [Export to Raw Data Storage](#5-simpan-ke-excel--csv)

---
> ℹ️ **Note:** Scraping data publik ditujukan untuk riset data analyst & open-source intelligence (OSINT).


## 1. Persiapan & Instalasi

### Apa itu Tweet Harvest?
**tweet-harvest** adalah CLI tool berbasis Node.js yang memanfaatkan akun Twitter/X yang sudah login untuk mengambil tweet menggunakan Advanced Search Twitter. Tool ini **tidak membutuhkan API key** dari Twitter, cukup menggunakan cookie sesi dari browser.

### Persyaratan Sistem
| Komponen | Versi Minimum | Keterangan |
|---|---|---|
| Node.js | ≥ 18.x | [Download Node.js](https://nodejs.org) |
| npm | ≥ 8.x | Biasanya sudah include dengan Node.js |
| Python | ≥ 3.8 | Untuk post-processing |
| Akun Twitter/X | Aktif | Wajib login via browser terlebih dahulu |

In [1]:
# ── Cek versi Node.js dan npm ──
import subprocess, sys

def run_cmd(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip()

node_v, _ = run_cmd("node --version")
npm_v,  _ = run_cmd("npm --version")

print(f"Node.js : {node_v if node_v else '❌ Tidak ditemukan — install dari https://nodejs.org'}")
print(f"npm     : {npm_v  if npm_v  else '❌ Tidak ditemukan'}")
print()

if not node_v:
    print("🔴 Node.js belum terinstall!")
    print("   → Windows : Download di https://nodejs.org/en/download")
    print("   → Mac     : brew install node")
    print("   → Linux   : sudo apt install nodejs npm")
else:
    major = int(node_v.lstrip('v').split('.')[0])
    if major < 18:
        print(f"⚠️  Node.js v{major} terlalu lama. Butuh ≥ v18. Update di https://nodejs.org")
    else:
        print("✅ Node.js sudah memenuhi syarat!")


Node.js : v25.8.0
npm     : 11.12.1

✅ Node.js sudah memenuhi syarat!


In [2]:
# ── Install tweet-harvest secara global ──
import subprocess

print("📦 Menginstall tweet-harvest...")
result = subprocess.run(
    "npm install -g tweet-harvest",
    shell=True, capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ tweet-harvest berhasil diinstall!")
    ver_out, _ = run_cmd("tweet-harvest --version 2>/dev/null || npx tweet-harvest --version")
    print(f"   Versi : {ver_out}")
else:
    print("❌ Gagal install. Coba jalankan sebagai admin / sudo:")
    print("   sudo npm install -g tweet-harvest")
    print()
    print("Stderr:", result.stderr[:500])


📦 Menginstall tweet-harvest...
✅ tweet-harvest berhasil diinstall!
   Versi : Tweet Harvest [v2.7.1]

Research by Helmi Satria
Use it for Educational Purposes only!

This script uses Chromium Browser to crawl data from Twitter with your Twitter auth token.
Please enter your Twitter auth token when prompted.

Note: Keep your access token secret! Don't share it with anyone else.
Note: This script only runs on your local device.

2.7.1


## 2. Login & Cookie Setup

### Mengapa Butuh Cookie?
tweet-harvest bekerja layaknya browser yang sudah login ke Twitter/X. Ia membutuhkan cookie sesi (`auth_token`) agar Twitter mengizinkan akses ke Advanced Search.

### Cara Mendapatkan Cookie `auth_token`

**Metode A — Export Cookie Otomatis (Rekomendasi)**
1. Install ekstensi browser **EditThisCookie** (Chrome) atau **Cookie Quick Manager** (Firefox)
2. Buka [twitter.com](https://twitter.com) dan pastikan sudah login
3. Klik ekstensi → Export cookie → cari nilai `auth_token`

**Metode B — DevTools Manual**
1. Buka Twitter/X di browser, login
2. Tekan `F12` → tab **Application** (Chrome) atau **Storage** (Firefox)
3. Pilih **Cookies** → `https://twitter.com`
4. Cari cookie bernama `auth_token` → copy nilainya

```
Contoh auth_token: abc123def456ghi789...  (panjang ~40 karakter)
```

> ⚠️ **PENTING:** Jangan bagikan `auth_token` ke siapapun — ini setara dengan password!


In [ ]:
# ─── SEL KHUSUS AUTENTIKASI SCRAPING ─────────────────────────────────
import os
import getpass
from pathlib import Path

# Ambil token dari environment variable atau input interaktif aman via getpass
AUTH_TOKEN = os.environ.get("TWITTER_AUTH_TOKEN")

if not AUTH_TOKEN or AUTH_TOKEN == "AUTH_TOKEN" or AUTH_TOKEN == "GANTI_DENGAN_AUTH_TOKEN_KAMU":
    AUTH_TOKEN = getpass.getpass("Masukkan Twitter/X auth_token (tidak akan terlihat di layar): ").strip()

if AUTH_TOKEN and len(AUTH_TOKEN) > 20:
    masked = AUTH_TOKEN[:6] + "..." + AUTH_TOKEN[-4:]
    print(f"Auth token berhasil dimuat ke runtime: {masked} ({len(AUTH_TOKEN)} karakter)")
else:
    print("Auth token belum diisi. Pastikan token diinput sebelum menjalankan scraping.")


✅ auth_token ditemukan: 85a9c6...0225 (panjang: 40 karakter)


## 3. Scraping Tweet

### Parameter Penting tweet-harvest
| Parameter | Keterangan | Contoh |
|---|---|---|
| `--token` | auth_token dari cookie | `abc123...` |
| `--search-keyword` | Query pencarian Twitter Advanced Search | `ww3 iran` |
| `--limit` | Jumlah maksimal tweet yang diambil | `100` |
| `--output` | Nama file output (`.csv` otomatis ditambahkan) | `hasil_ww3` |
| `--delay` | Jeda antar request (ms) | `1000` |

### Strategi Keyword
Keyword yang digunakan mengikuti sintaks **Twitter Advanced Search**:
- `OR` → salah satu harus ada
- `()` → grouping logika
- `lang:en` → filter bahasa Inggris
- `-filter:retweets` → abaikan retweet

In [4]:
# ── Konfigurasi Scraping ──
import subprocess, os, json
from pathlib import Path
from datetime import datetime

# ─── PARAMETER UTAMA ────────────────────────────────
KEYWORD = '("world war" OR ww3 OR wwiii) (iran OR tehran) (america OR us OR usa OR washington) lang:en -filter:retweets'
LIMIT   = 200          # Jumlah tweet (max ~1000 per sesi untuk menghindari rate-limit)
OUTPUT  = "dataset_WW3_tweetharvest"
DELAY   = 1500         # Delay antar request dalam ms (1500ms = aman dari rate-limit)
# ────────────────────────────────────────────────────

OUTPUT_DIR = Path("data/raw")
OUTPUT_DIR.mkdir(exist_ok=True)

output_path = OUTPUT_DIR / OUTPUT

import sys
cmd_base = "npx.cmd" if sys.platform == "win32" else "npx"

cmd = [
    cmd_base, "tweet-harvest",
    "--token",          AUTH_TOKEN,
    "--search-keyword", KEYWORD,
    "--limit",          str(LIMIT),
    "--output",         str(output_path),
    "--delay",          str(DELAY),
]


print("=" * 60)
print("  KONFIGURASI SCRAPING TWEET HARVEST")
print("=" * 60)
print(f"  Keyword : {KEYWORD[:60]}...")
print(f"  Limit   : {LIMIT} tweet")
print(f"  Output  : {output_path}.csv")
print(f"  Delay   : {DELAY}ms antar request")
print("=" * 60)
print()

# Preview perintah (tanpa auth_token)
safe_cmd = ' '.join(cmd).replace(AUTH_TOKEN, '***TOKEN***')
print(f"Perintah: {safe_cmd}")


  KONFIGURASI SCRAPING TWEET HARVEST
  Keyword : ("world war" OR ww3 OR wwiii) (iran OR tehran) (america OR u...
  Limit   : 200 tweet
  Output  : output_tweetharvest\dataset_WW3_tweetharvest.xlsx
  Delay   : 1500ms antar request

Perintah: npx.cmd tweet-harvest --token ***TOKEN*** --search-keyword ("world war" OR ww3 OR wwiii) (iran OR tehran) (america OR us OR usa OR washington) lang:en -filter:retweets --limit 200 --output output_tweetharvest\dataset_WW3_tweetharvest --delay 1500


In [5]:
# ── Scraping Bertahap (Strategi Anti Rate-Limit) ──
import subprocess
import time as _t
import sys
from pathlib import Path

# Definisikan folder output khusus untuk tweet-harvest
OUTPUT_DIR = Path("data/raw") # tweet-harvest v2.x otomatis menggunakan folder ini atau di dalam project

BATCH_CONFIG = [
    {
        "desc"    : "Batch 1 — Kata kunci WW3 Iran",
        "keyword" : '("world war 3" OR ww3 OR wwiii) iran lang:en -filter:retweets',
        "limit"   : 100,
        "output"  : "batch1_ww3_iran", # Hanya nama file
    },
    {
        "desc"    : "Batch 2 — Kata kunci America US",
        "keyword" : '("world war 3" OR ww3 OR wwiii) (america OR usa) lang:en -filter:retweets',
        "limit"   : 100,
        "output"  : "batch2_ww3_us",
    },
    {
        "desc"    : "Batch 3 — Kata kunci Tehran Washington",
        "keyword" : '("world war" OR wwiii) (tehran OR washington) lang:en -filter:retweets',
        "limit"   : 100,
        "output"  : "batch3_tehran_washington",
    },
]

cmd_base = "npx.cmd" if sys.platform == "win32" else "npx"

for i, batch in enumerate(BATCH_CONFIG, 1):
    print(f"\n{'='*50}")
    print(f"  {batch['desc']}")
    print(f"{'='*50}")

    cmd_batch = [
        cmd_base, "tweet-harvest",
        "--token",          AUTH_TOKEN,
        "--search-keyword", batch["keyword"],
        "--limit",          str(batch["limit"]),
        "-o",               batch["output"],
        "-e",               "csv", # Memastikan format export ke CSV
        "--delay",          "3",   # Waktu delay 3 detik (bukan ms)
    ]
    
    print(f"  ⏳ Menjalankan Batch {i}...")
    
    result = subprocess.run(cmd_batch, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  ✅ Batch {i} selesai!")
    else:
        # Jika gagal, tampilkan error dan hentikan seluruh proses scraping
        print(f"  ❌ Batch {i} gagal dengan exit code {result.returncode}!")
        print(f"  Pesan Error:\n{result.stderr[-500:]}")
        print("\n🚨 Menghentikan proses scraping karena terjadi error. Kode tidak akan melanjutkan ke batch berikutnya.")
        break
    
    if i < len(BATCH_CONFIG):
        wait = 15
        print(f"  ⏳ Tunggu {wait} detik sebelum lanjut ke batch berikutnya...")
        _t.sleep(wait)

print("\nProses batch scraping selesai/dihentikan.")



  Batch 1 — Kata kunci WW3 Iran
  ⏳ Menjalankan Batch 1...
  ✅ Batch 1 selesai!
  ⏳ Tunggu 15 detik sebelum lanjut ke batch berikutnya...

  Batch 2 — Kata kunci America US
  ⏳ Menjalankan Batch 2...
  ✅ Batch 2 selesai!
  ⏳ Tunggu 15 detik sebelum lanjut ke batch berikutnya...

  Batch 3 — Kata kunci Tehran Washington
  ⏳ Menjalankan Batch 3...
  ✅ Batch 3 selesai!

Proses batch scraping selesai/dihentikan.


## 4. Load & Validasi Data

In [6]:
# ── Load Data Hasil Scraping & Gabungkan ──
import pandas as pd
import numpy as np
from pathlib import Path

# Folder di mana tweet-harvest menyimpan file
# (Biasanya di direktori utama atau di folder "tweets-data")
DATA_DIR = Path("tweets-data")

# Cari semua file CSV hasil scraping di folder
csv_files = list(DATA_DIR.glob("*.csv"))

if not csv_files:
    print("⚠️ Belum ada file CSV yang ditemukan.")
else:
    dfs = []
    for f in csv_files:
        try:
            # Baca tiap file CSV dan tambahkan info dari file mana ia berasal
            _df = pd.read_csv(f, delimiter=";", dtype=str) # tweet-harvest kadang menggunakan delimiter ';'
            _df['source_file'] = f.name
            dfs.append(_df)
            print(f"  ✅ {f.name} — {len(_df):,} baris")
        except Exception as e:
            # Fallback membaca menggunakan comma (,) jika semicolon (;) gagal
            try:
                _df = pd.read_csv(f, dtype=str)
                _df['source_file'] = f.name
                dfs.append(_df)
                print(f"  ✅ {f.name} — {len(_df):,} baris")
            except Exception as e2:
                print(f"  ❌ Gagal membaca {f.name} — Error: {e2}")

    if dfs:
        # 1. Gabungkan Semua File Menjadi Satu
        df_combined = pd.concat(dfs, ignore_index=True)
        
        # 2. Hapus duplikat berdasarkan ID Tweet agar tidak ada tweet ganda antar batch
        if 'id' in df_combined.columns:
            df_combined = df_combined.drop_duplicates(subset=['id'])
            
        print(f"\n📦 Total gabungan tweet unik: {len(df_combined):,} baris")
        
        # 3. Simpan ke SATU file CSV final
        final_csv_path = "WW3_Iran_US_Combined.csv"
        df_combined.to_csv(final_csv_path, index=False, encoding='utf-8-sig')
        print(f"🎉 Semua data berhasil digabung dan disimpan ke: {final_csv_path}")
        
        # Set dataframe utama (jika cell selanjutnya bergantung pada variabel 'df')
        df = df_combined


  ✅ batch1_ww3_iran.csv — 26 baris
  ✅ batch2_ww3_us.csv — 80 baris
  ✅ batch3_tehran_washington.csv — 100 baris

📦 Total gabungan tweet unik: 206 baris
🎉 Semua data berhasil digabung dan disimpan ke: WW3_Iran_US_Combined.csv


In [7]:
# ── Eksplorasi & Mapping Kolom ──
# tweet-harvest menghasilkan kolom dengan nama standar

if df is not None and len(df) > 0:
    print("=" * 60)
    print("  INFORMASI DATASET")
    print("=" * 60)
    print(f"  Jumlah tweet     : {len(df):,}")
    print(f"  Jumlah kolom     : {len(df.columns)}")
    print(f"  Kolom tersedia   : {df.columns.tolist()}")
    print()
    print(df.dtypes)
    print()
    print("--- Preview 3 baris pertama ---")
    display(df.head(3))
else:
    # Demo dengan struktur kolom yang diharapkan dari tweet-harvest
    print("📋 Struktur kolom yang dihasilkan tweet-harvest:")
    expected_cols = {
        'full_text'         : 'Isi tweet',
        'created_at'        : 'Waktu posting',
        'user_screen_name'  : 'Username (@handle)',
        'user_name'         : 'Nama display',
        'user_followers_count': 'Jumlah follower',
        'user_friends_count'  : 'Jumlah following',
        'retweet_count'     : 'Jumlah retweet',
        'favorite_count'    : 'Jumlah like',
        'reply_count'       : 'Jumlah reply',
        'quote_count'       : 'Jumlah quote tweet',
        'lang'              : 'Bahasa tweet',
        'tweet_url'         : 'URL tweet',
        'id_str'            : 'ID unik tweet',
    }
    for col, desc in expected_cols.items():
        print(f"  {col:<30} → {desc}")


  INFORMASI DATASET
  Jumlah tweet     : 206
  Jumlah kolom     : 17
  Kolom tersedia   : ['conversation_id_str,"created_at","favorite_count","full_text","id_str","image_url","in_reply_to_screen_name","lang","location","quote_count","reply_count","retweet_count","tweet_url","user_id_str","username"', 'source_file', 'conversation_id_str', 'created_at', 'favorite_count', 'full_text', 'id_str', 'image_url', 'in_reply_to_screen_name', 'lang', 'location', 'quote_count', 'reply_count', 'retweet_count', 'tweet_url', 'user_id_str', 'username']

conversation_id_str,"created_at","favorite_count","full_text","id_str","image_url","in_reply_to_screen_name","lang","location","quote_count","reply_count","retweet_count","tweet_url","user_id_str","username"    str
source_file                                                                                                                                                                                                        str
conversation_id_str        

,"conversation_id_str,""created_at"",""favorite_count"",""full_text"",""id_str"",""image_url"",""in_reply_to_screen_name"",""lang"",""location"",""quote_count"",""reply_count"",""retweet_count"",""tweet_url"",""user_id_str"",""username""",source_file,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
0,"2049911716583019008,""2026-04-30T18:00:13.000Z""...",batch1_ww3_iran.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"🇺🇸🇮🇱🇮🇷‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️‼️"",""2049911716583...",batch1_ww3_iran.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"2050709898590126224,""2026-05-02T22:51:54.000Z""...",batch1_ww3_iran.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# ── Pembersihan & Standarisasi Kolom ──
# Mapping kolom dari tweet-harvest CSV ke format standar
COLUMN_MAP = {
    'full_text'           : 'tweet',
    'created_at'          : 'timestamp',
    'username'            : 'username', 
    'user_screen_name'    : 'username', # Fallback
    'retweet_count'       : 'retweets',
    'favorite_count'      : 'likes',
    'reply_count'         : 'replies',
    'quote_count'         : 'quotes',
    'lang'                : 'language',
    'tweet_url'           : 'url',
    'id_str'              : 'tweet_id',
}

if df is not None and len(df) > 0:
    # Rename kolom yang ada
    df_clean = df.rename(columns={k: v for k, v in COLUMN_MAP.items() if k in df.columns})

    # Hapus duplikat berdasarkan tweet_id
    before = len(df_clean)
    if 'tweet_id' in df_clean.columns:
        df_clean = df_clean.drop_duplicates(subset=['tweet_id'])
    after = len(df_clean)
    print(f"✅ Duplikat dihapus : {before - after} baris")

    # Konversi timestamp ke format datetime yang benar
    if 'timestamp' in df_clean.columns:
        df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'], errors='coerce')
        df_clean['date']  = df_clean['timestamp'].dt.date
        df_clean['hour']  = df_clean['timestamp'].dt.hour
        
    # PENTING: Konversi kolom metrik kembali menjadi angka (integer)
    for col in ['likes', 'retweets', 'replies', 'quotes']:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0).astype(int)

    # Tambah kolom keyword
    df_clean['keyword'] = 'WW3-Iran-US'

    print(f"✅ Dataset bersih  : {len(df_clean):,} tweet\n")
    display(df_clean.head(3))
else:
    print("Dataset kosong.")
    df_clean = pd.DataFrame()


✅ Duplikat dihapus : 25 baris
✅ Dataset bersih  : 181 tweet



,"conversation_id_str,""created_at"",""favorite_count"",""full_text"",""id_str"",""image_url"",""in_reply_to_screen_name"",""lang"",""location"",""quote_count"",""reply_count"",""retweet_count"",""tweet_url"",""user_id_str"",""username""",source_file,conversation_id_str,timestamp,likes,tweet,tweet_id,image_url,in_reply_to_screen_name,language,location,quotes,replies,retweets,url,user_id_str,username,date,hour,keyword
0,"2049911716583019008,""2026-04-30T18:00:13.000Z""...",batch1_ww3_iran.csv,NaN,NaT,0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN,NaN,NaT,NaN,WW3-Iran-US
26,NaN,batch2_ww3_us.csv,2053331698629370273,2026-05-10 04:30:00+00:00,79,🇺🇸🌍 Why the UAE Crisis Could Change America’s ...,2053331698629370273,https://pbs.twimg.com/amplify_video_thumb/2053...,NaN,en,NaN,0,2,14,https://x.com/Worldwar_3_/status/2053331698629...,1465438438879019021,Worldwar_3_,2026-05-10,4.0,WW3-Iran-US
27,NaN,batch2_ww3_us.csv,2052363825303990440,2026-05-07 12:24:01+00:00,132,"""Professor Jiang: World War 3 Has Already Begu...",2052363825303990440,https://pbs.twimg.com/amplify_video_thumb/2052...,NaN,en,NaN,6,3,34,https://x.com/_pblanknews/status/2052363825303...,1369489961238417410,_pblanknews,2026-05-07,12.0,WW3-Iran-US


## 5. Simpan ke Excel & CSV

In [17]:
# ── Simpan Dataset Final ──
import os
from datetime import datetime
from pathlib import Path

# Kita selaraskan foldernya ke direktori output yang baru
OUTPUT_DIR = Path("data/raw") 
OUTPUT_DIR.mkdir(exist_ok=True)

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")

if df_clean is not None and len(df_clean) > 0:

    # ─── Simpan CSV ──────────────────────────────────────────
    csv_path = OUTPUT_DIR / f"dataset_WW3_Final_{timestamp_str}.csv"
    df_clean.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"✅ CSV tersimpan secara final   : {csv_path}")

    print(f"\n📦 Total baris disimpan : {len(df_clean):,}")

else:
    print("Dataset kosong — tidak ada yang disimpan.")


✅ CSV tersimpan secara final   : tweets-data\dataset_WW3_Final_20260510_2300.csv

📦 Total baris disimpan : 181
